In [1]:
# Что делает этот ноутбук:
# 1. Загружает очищенные данные из artifacts (результаты EDA)
# 2. Добавляет Feature Engineering (логи, бинаризация)
# 3. Добавляет признаки из токенов (submit.csv)
# 4. Обучает LightGBM и CatBoost с кросс-валидацией
# 5. Делает блендинг моделей
# 6. Сохраняет сабмишен

In [2]:
import sys
!{sys.executable} -m pip install scikit-learn lightgbm catboost scipy

  Using cached scikit_learn-1.8.0-cp312-cp312-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached lightgbm-4.6.0-py3-none-macosx_12_0_arm64.whl.metadata (17 kB)
  Using cached catboost-1.2.10-cp312-cp312-macosx_11_0_universal2.whl.metadata (1.4 kB)
  Using cached scipy-1.17.1-cp312-cp312-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
  Using cached plotly-6.7.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached narwhals-2.21.2-py3-none-any.whl.metadata (16 kB)
Using cached scikit_learn-1.8.0-cp312-cp312-macosx_12_0_arm64.whl (8.1 MB)
Using cached lightgbm-4.6.0-py3-none-macosx_12_0_arm64.whl (1.6 MB)
Using cached catboost-1.2.10-cp312-cp312-macosx_11_0_universal2.whl (28.9 MB)
Using cached scipy-1.17.1-cp312-cp312-macosx_14_0_arm64.whl (20.3 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Usin

In [ ]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

02_MODELLING


In [4]:
# 1. ЗАГРУЗКА АРТЕФАКТОВ ИЗ EDA
ARTIFACTS_DIR = Path("artifacts")

X_train = pd.read_parquet(ARTIFACTS_DIR / "X_train_v1.parquet")
X_test = pd.read_parquet(ARTIFACTS_DIR / "X_test_v1.parquet")
y = pd.read_parquet(ARTIFACTS_DIR / "y_train.parquet")["target"]

with open(ARTIFACTS_DIR / "feature_groups.pkl", "rb") as f:
    groups = pickle.load(f)

feature_summary = pd.read_parquet(ARTIFACTS_DIR / "feature_summary.parquet")
nonzero_effect = pd.read_parquet(ARTIFACTS_DIR / "nonzero_effect.parquet")
minus_one_effect = pd.read_parquet(ARTIFACTS_DIR / "minus_one_effect.parquet")

print(f"   X_train shape: {X_train.shape}")
print(f"   X_test shape: {X_test.shape}")
print(f"   Target rate: {y.mean():.6f}")
print(f"   Всего признаков: {X_train.shape[1]}")

   X_train shape: (247972, 1360)
   X_test shape: (106274, 1360)
   Target rate: 0.013493
   Всего признаков: 1360


In [5]:
# 2. FEATURE ENGINEERING (на основе EDA)

# 2.1 Бинаризация информативных -1 (всего 2 признака)
informative_m1 = groups['informative_m1']
print(f"   Бинаризация -1 для {len(informative_m1)} признаков: {informative_m1}")
for col in informative_m1:
    X_train[f'{col}_is_m1'] = (X_train[col] == -1).astype(int)
    X_test[f'{col}_is_m1'] = (X_test[col] == -1).astype(int)

# 2.2 Бинаризация ненулевых значений (топ-30 по эффекту)
top_nonzero = nonzero_effect.head(30)['feature'].tolist()
print(f"   Бинаризация ненулевых для {len(top_nonzero)} признаков")
for col in top_nonzero:
    X_train[f'{col}_nonzero'] = (X_train[col] != 0).astype(int)
    X_test[f'{col}_nonzero'] = (X_test[col] != 0).astype(int)

# 2.3 Логарифмирование скошенных признаков (топ-50)
skewed_cols = feature_summary[feature_summary['is_high_skew']] \
    .sort_values('abs_corr_target', ascending=False).head(50).index
print(f"   Логарифмирование для {len(skewed_cols)} скошенных признаков")
for col in skewed_cols:
    shift = abs(X_train[col].min()) + 1e-5
    X_train[f'{col}_log'] = np.log1p(X_train[col] + shift)
    X_test[f'{col}_log'] = np.log1p(X_test[col] + shift)

print(f"   После FE: {X_train.shape[1]} признаков")

   Бинаризация -1 для 2 признаков: ['feature_1119', 'feature_1110']
   Бинаризация ненулевых для 30 признаков
   Логарифмирование для 50 скошенных признаков
   После FE: 1442 признаков


In [6]:
# 3. ДОБАВЛЕНИЕ ПРИЗНАКОВ ИЗ ТОКЕНОВ (submit.csv)
tokens = pd.read_csv('submit.csv')
print(f"   Tokens shape: {tokens.shape}")

# Агрегируем статистики
token_features = tokens.groupby('index')['score'].agg([
    ('token_count', 'count'),
    ('token_sum', 'sum'),
    ('token_mean', 'mean'),
    ('token_std', 'std'),
    ('token_max', 'max'),
]).reset_index()

# Энтропия
def entropy(x):
    probs = x / x.sum()
    return -(probs * np.log(probs + 1e-10)).sum()

token_entropy = tokens.groupby('index')['score'].agg([('token_entropy', entropy)]).reset_index()
token_features = token_features.merge(token_entropy, on='index', how='left').fillna(0)
token_features['has_tokens'] = (token_features['token_count'] > 0).astype(int)

# Присоединяем (по индексу строки, так как X_train и X_test имеют индекс = user_id)
X_train = X_train.merge(token_features, left_index=True, right_index=True, how='left').fillna(0)
X_test = X_test.merge(token_features, left_index=True, right_index=True, how='left').fillna(0)

print(f"   После токенов: {X_train.shape[1]} признаков")

   Tokens shape: (106274, 2)
   После токенов: 1450 признаков


In [ ]:
# 4. ОБРАБОТКА ПРОПУСКОВ (если остались)
from sklearn.impute import SimpleImputer

missing_train = X_train.isnull().sum().sum()
missing_test = X_test.isnull().sum().sum()
print(f"   Пропусков в train: {missing_train}")
print(f"   Пропусков в test: {missing_test}")

if missing_train > 0 or missing_test > 0:
    # Создаём индикаторы пропусков
    for col in X_train.columns:
        if X_train[col].isnull().any():
            X_train[f'{col}_was_missing'] = X_train[col].isnull().astype(int)
            X_test[f'{col}_was_missing'] = X_test[col].isnull().astype(int)
    
    # Заполняем медианой
    imputer = SimpleImputer(strategy='median')
    X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
    X_test = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)
    print("   Пропуски заполнены медианой")

print(f"   Финальный размер: X_train={X_train.shape}, X_test={X_test.shape}")



4. Обработка пропусков...
   Пропусков в train: 0
   Пропусков в test: 0
   Финальный размер: X_train=(247972, 1450), X_test=(106274, 1450)


In [ ]:
# 5. Обучение модели LIGHTGBM
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# Параметры модели
params_lgb = {
    'objective': 'binary',
    'metric': 'auc',
    'num_leaves': 31,
    'max_depth': 7,
    'learning_rate': 0.03,
    'feature_fraction': 0.7,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'n_estimators': 2000,
    'random_state': 42,
    'verbose': -1,
    'is_unbalance': True,  # балансировка классов
    'n_jobs': -1
}

# Кросс-валидация
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_lgb = np.zeros(len(X_train))
test_lgb = np.zeros(len(X_test))
lgb_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y)):
    print(f"\n   Fold {fold+1}/5")
    
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = lgb.LGBMClassifier(**params_lgb)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',
        callbacks=[lgb.early_stopping(100, verbose=False)]
    )
    
    oof_lgb[val_idx] = model.predict_proba(X_val)[:, 1]
    test_lgb += model.predict_proba(X_test)[:, 1] / 5
    
    auc = roc_auc_score(y_val, oof_lgb[val_idx])
    lgb_scores.append(auc)
    print(f"      AUC: {auc:.6f}")

print(f"\n   LightGBM OOF AUC: {roc_auc_score(y, oof_lgb):.6f}")
print(f"   LightGBM mean AUC: {np.mean(lgb_scores):.6f} ± {np.std(lgb_scores):.6f}")


5. ОБУЧЕНИЕ LIGHTGBM

   Fold 1/5
      AUC: 0.654296

   Fold 2/5
      AUC: 0.659442

   Fold 3/5
      AUC: 0.640583

   Fold 4/5
      AUC: 0.657675

   Fold 5/5
      AUC: 0.648974

   LightGBM OOF AUC: 0.650228
   LightGBM mean AUC: 0.652194 ± 0.006814


In [9]:
# 6. Обучение модели CATBOOST
try:
    from catboost import CatBoostClassifier
    
    params_cb = {
        'iterations': 1500,
        'learning_rate': 0.03,
        'depth': 6,
        'eval_metric': 'AUC',
        'random_seed': 42,
        'verbose': 500,
        'early_stopping_rounds': 100,
        'auto_class_weights': 'Balanced'
    }
    
    oof_cb = np.zeros(len(X_train))
    test_cb = np.zeros(len(X_test))
    cb_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y)):
        print(f"\n   Fold {fold+1}/5")
        
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = CatBoostClassifier(**params_cb)
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True, verbose=False)
        
        oof_cb[val_idx] = model.predict_proba(X_val)[:, 1]
        test_cb += model.predict_proba(X_test)[:, 1] / 5
        
        auc = roc_auc_score(y_val, oof_cb[val_idx])
        cb_scores.append(auc)
        print(f"      AUC: {auc:.6f}")
    
    print(f"\n   CatBoost OOF AUC: {roc_auc_score(y, oof_cb):.6f}")
    print(f"   CatBoost mean AUC: {np.mean(cb_scores):.6f} ± {np.std(cb_scores):.6f}")
    
    use_catboost = True
except ImportError:
    print("\n   ⚠️ CatBoost не установлен, пропускаем...")
    use_catboost = False


   Fold 1/5
      AUC: 0.659531

   Fold 2/5
      AUC: 0.672876

   Fold 3/5
      AUC: 0.644557

   Fold 4/5
      AUC: 0.663121

   Fold 5/5
      AUC: 0.665157

   CatBoost OOF AUC: 0.659336
   CatBoost mean AUC: 0.661048 ± 0.009331


In [10]:
# 7. БЛЕНДИНГ МОДЕЛЕЙ
if use_catboost:
    from scipy.optimize import minimize
    
    def optimize_weights(oof1, oof2, y_true):
        def objective(w):
            blend = w[0] * oof1 + w[1] * oof2
            return -roc_auc_score(y_true, blend)
        
        result = minimize(objective, [0.5, 0.5], bounds=[(0,1), (0,1)],
                          constraints={'type': 'eq', 'fun': lambda w: w[0] + w[1] - 1})
        return result.x
    
    weights = optimize_weights(oof_lgb, oof_cb, y)
    print(f"\n   Оптимальные веса:")
    print(f"      LightGBM: {weights[0]:.4f}")
    print(f"      CatBoost: {weights[1]:.4f}")
    
    final_test = weights[0] * test_lgb + weights[1] * test_cb
else:
    print("\n   Используем только LightGBM")
    final_test = test_lgb


   Оптимальные веса:
      LightGBM: 0.5000
      CatBoost: 0.5000


In [11]:
# 8. СОХРАНЕНИЕ САБМИШЕНА
test_ids = pd.read_parquet(ARTIFACTS_DIR / "test_ids.parquet")

submission = pd.DataFrame({
    'index': test_ids['index'].values,
    'score': final_test
})
submission.to_csv('submission_final.csv', index=False)

print(f"\n   ✅ Сабмишен сохранён: submission_final.csv")
print(f"\n   Первые 5 строк:")
print(submission.head())

print("\n✅ ГОТОВО! Можете отправлять submission_final.csv на Kaggle.")


   ✅ Сабмишен сохранён: submission_final.csv

   Первые 5 строк:
    index     score
0  194357  0.217906
1  313222  0.419006
2  321873  0.531477
3  118689  0.351339
4  342561  0.170591

✅ ГОТОВО! Можете отправлять submission_final.csv на Kaggle.
